# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/J-Aamir/Capstone-Flyrank/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import os
if not os.path.exists("Capstone-Flyrank"):
    !git clone https://github.com/J-Aamir/Capstone-Flyrank.git
%cd Capstone-Flyrank

import pandas as pd, numpy as np, json
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

os.makedirs('work/figures', exist_ok=True)
os.makedirs('work/outputs', exist_ok=True)


## 1. Question

**Research question:** out of thousands of content pages, which ones should an editor review first, because they are losing search visibility.

**The decision it supports:** a content or SEO editor has limited time each week, usually enough to review twenty to fifty pages. They need a ranked list, not a yes or no answer on every page. A wrong pick costs either wasted editor hours on a page that was fine, or a missed page that kept losing visibility unnoticed.

In [ ]:
print("Research question: which pages should be reviewed first for a content refresh?")
print("Decision supported: weekly editorial triage, roughly 20-50 pages per review cycle.")


## 2. Data

We use the starter release of the FlyRank ML Internship dataset: 30,000 anonymized content pages across 32 clients, 44 columns of trailing 90 day search and analytics metrics. Every page and client identifier is pseudonymized, so nothing here points back to a real company or URL.

**Excluded, and why:**
- `trend_direction` and `trend_pct`, because these two columns define our label. Using them as model inputs would let the model see the answer directly.
- Rows where `avg_position` is zero, because in this dataset a zero means no ranking data was recorded, not that the page ranked first.
- Any text that could identify a client or page beyond the pseudonymized IDs.

In [ ]:
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
print('Rows, columns:', df.shape)
print('Clients:', df.client_id.nunique())

df['is_declining_label'] = (df.trend_direction == 'down').astype(int)
d = df[df.avg_position > 0].copy()
print('Rows after dropping avg_position == 0 (no data):', len(d), 'of', len(df))
print('Excluded from features:', ['trend_direction', 'trend_pct'])


## 3. Methodology

**Label:** a page is declining if impressions fell 20 percent or more against the prior 30 day window. This is a proxy built from a current window comparison, not a verified future outcome.

**Baseline:** a hand written rule flags a page if its click through rate sits well below the average for pages ranking at a similar position, weighted by impressions. Reason code: `ctr_below_position_benchmark`.

**Model:** a Random Forest trained on seven features that are all knowable in advance: content age, days since last update, impressions, average position, click through rate, word count, and search volume.

**Validation design:** split by client, not by row, so no client's pages appear in both train and test. We also ran the same model under a naive row random split to show the difference it makes.

**Leakage check:** confirm none of the seven features overlap with the two label-source columns, and check no feature is suspiciously highly correlated with the label.

In [ ]:
features = ['content_age_days','days_since_last_update','impressions_90d','avg_position',
            'ctr','word_count','search_volume']

# leakage check
label_source_cols = ['trend_direction', 'trend_pct']
overlap = set(features) & set(label_source_cols)
assert len(overlap) == 0, 'Leakage detected'
print('Leakage check passed. Features:', features)

corrs = d[features + ['is_declining_label']].corr()['is_declining_label'].drop('is_declining_label')
print()
print('Feature correlation with label (flag anything near +-1.0):')
print(corrs.round(3).sort_values())


## 4. Results (vs baseline)

Same client grouped test split, same metric, for both the baseline rule and the model.

In [ ]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

clients = d['client_id'].unique()
train_clients, test_clients = train_test_split(clients, test_size=0.3, random_state=42)
train = d[d.client_id.isin(train_clients)]
test  = d[d.client_id.isin(test_clients)].copy()

# baseline, benchmark learned from train only
benchmark = train.groupby('position_tier')['ctr'].mean()
test['benchmark_ctr'] = test['position_tier'].map(benchmark)
test['ctr_gap'] = (test['benchmark_ctr'] - test['ctr']).clip(lower=0)
test['baseline_score'] = test['ctr_gap'] * test['impressions_90d']

# model
Xtr, Xte = train[features].fillna(0), test[features].fillna(0)
ytr, yte = train['is_declining_label'].values, test['is_declining_label'].values
rf = RandomForestClassifier(n_estimators=300, max_depth=6, class_weight='balanced',
                             random_state=42, n_jobs=-1).fit(Xtr, ytr)
test['model_score'] = rf.predict_proba(Xte)[:, 1]

results = []
for k in (20, 50):
    results.append({
        'k': k,
        'baseline': round(precision_at_k(test['baseline_score'].values, yte, k), 3),
        'random_forest': round(precision_at_k(test['model_score'].values, yte, k), 3),
    })
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

# honest-split comparison, also shown in the paper's methodology section
Xtr_r, Xte_r, ytr_r, yte_r = train_test_split(d[features].fillna(0), d['is_declining_label'], test_size=0.3, random_state=42)
rf_r = RandomForestClassifier(n_estimators=300, max_depth=6, class_weight='balanced', random_state=42, n_jobs=-1).fit(Xtr_r, ytr_r)
scores_r = rf_r.predict_proba(Xte_r)[:, 1]
leak_p50 = round(precision_at_k(scores_r, yte_r.values, 50), 3)
client_overlap = len(set(d.loc[Xtr_r.index, 'client_id']) & set(d.loc[Xte_r.index, 'client_id']))
print()
print(f'Row-random split Precision@50 (dishonest): {leak_p50}  -- {client_overlap}/{d.client_id.nunique()} clients leaked')
print(f'Client-grouped split Precision@50 (honest): {results_df.loc[results_df.k==50, "random_forest"].values[0]}')


## 5. Limitations

- This is an observational study. Nothing here proves that refreshing a page causes it to recover, only that certain page traits are associated with decline in this data.
- The label is a proxy built from a 30 day impressions comparison, not a confirmed future outcome.
- The model was trained and tested on 32 clients. We have not checked how it performs on a brand new client or industry.
- A Precision@50 of 0.74 means roughly one in four flagged pages will not actually be declining. This is decision support, not an automated judgment.

In [ ]:
limitations = [
    'Observational study: correlation, not proof of causation',
    'Label is a proxy from a 30-day window comparison, not a confirmed future outcome',
    'Validated on 32 clients only, not tested on new clients or industries',
    'Precision@50 of 0.74 means about 1 in 4 flagged pages will be a false alarm',
]
for item in limitations:
    print('LIMITATION:', item)


## 6. Ranked recommendations

Each test page gets one of four reason codes, combining the model's decline risk with the baseline's CTR gap signal.

In [ ]:
def assign(row):
    if row.model_score >= 0.6 and row.ctr_gap > 0.3:
        return 'high_decline_risk_and_ctr_gap', 'refresh_content_and_review_snippet'
    elif row.model_score >= 0.6:
        return 'high_decline_risk', 'refresh_content'
    elif row.ctr_gap > 0.3:
        return 'ctr_below_position_benchmark', 'review_title_and_meta_description'
    else:
        return 'low_priority', 'monitor_only'

codes = test.apply(assign, axis=1, result_type='expand')
test['reason_code'] = codes[0]
test['action'] = codes[1]
test['priority_score'] = test['model_score'] * 0.6 + (test['ctr_gap'] / test['ctr_gap'].max()) * 0.4

queue = test.sort_values('priority_score', ascending=False)
print('Reason code counts:')
print(queue['reason_code'].value_counts())
print()
print('Top 10:')
print(queue.head(10)[['content_id','priority_score','reason_code','action']].to_string(index=False))

queue[['content_id','client_id','priority_score','reason_code','action','model_score',
       'ctr_gap','trend_direction']].to_csv('work/outputs/action_playbook_queue.csv', index=False)
print()
print('Written: work/outputs/action_playbook_queue.csv')


## 7. Artifacts the paper embeds

Generates the comparison chart and the metrics file the deployed paper references.

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))
x = np.arange(len(results_df))
width = 0.35
ax.bar(x - width/2, results_df['baseline'], width, label='Baseline rule')
ax.bar(x + width/2, results_df['random_forest'], width, label='Random Forest')
ax.set_xticks(x)
ax.set_xticklabels([f"Precision@{k}" for k in results_df['k']])
ax.set_ylim(0, 1)
ax.set_title('Model vs baseline, client-grouped holdout')
ax.legend()
plt.tight_layout()
plt.savefig('work/figures/capstone_model_vs_baseline.png', dpi=150)
print('Written: work/figures/capstone_model_vs_baseline.png')

metrics = {
    'precision_at_20': {'baseline': float(results_df.loc[results_df.k==20,'baseline'].values[0]),
                         'random_forest': float(results_df.loc[results_df.k==20,'random_forest'].values[0])},
    'precision_at_50': {'baseline': float(results_df.loc[results_df.k==50,'baseline'].values[0]),
                         'random_forest': float(results_df.loc[results_df.k==50,'random_forest'].values[0])},
    'row_random_precision_at_50_dishonest': leak_p50,
    'client_overlap_in_row_random_split': client_overlap,
    'n_total_clients': int(d.client_id.nunique()),
    'reason_code_counts': queue['reason_code'].value_counts().to_dict(),
}
with open('work/outputs/capstone_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('Written: work/outputs/capstone_metrics.json')
print(json.dumps(metrics, indent=2))


## ML-12: demo outline, social post, employer summary

The three closing pieces that turn this project into something you can actually show someone.

In [ ]:
demo_outline = """
5-MINUTE DEMO OUTLINE

1. The problem (30 sec): a company has thousands of pages and can only review a handful
   each week. Show the 17-page hand rule result as the starting point.
2. The data (30 sec): 30,000 anonymized pages, 32 clients, show the dataframe head.
3. The honest mistake (90 sec): show the row-random split scoring 0.92, then show the
   client-grouped split dropping to 0.74. Explain why the second number is the real one.
4. The result (60 sec): show the final comparison chart, baseline 0.58 vs model 0.74 at
   Precision@50.
5. The playbook (60 sec): show the top 10 ranked actions with reason codes, and name one
   thing that should never be automated.
6. Close (30 sec): point to the live paper URL and the repo.
"""

social_post = (
    "I spent 7 weeks building a model that ranks which web pages a team should fix first, "
    "using real anonymized search data from FlyRank. The honest version beats a hand written "
    "rule by a wide margin, but only once I caught myself testing it the wrong way first. "
    "Full writeup and code here: [paper URL]"
)

employer_summary = (
    "I built and validated a content-decline prediction model on a 30,000-row real-world SEO "
    "dataset, improving Precision@50 from 0.58 to 0.74 over a hand written baseline using a "
    "properly client-grouped holdout split. Along the way I caught and corrected a data leakage "
    "issue that had inflated an earlier version of the same model to a misleading 0.92, and I "
    "turned the result into a ranked action playbook with clear human-review rules. The full "
    "paper, code, and notebooks are public."
)

print(demo_outline)
print()
print('SOCIAL POST:')
print(social_post)
print()
print('EMPLOYER SUMMARY:')
print(employer_summary)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
